In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## CONFIG AND EXTRACT RAE DATA

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations


BASE = "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/Steganalysis/logs"

METHODS = {
    "M1": "MUSDB_LSB_1_NoRandom/CNN_D5_F64_LR0.0001_20260611-181303",
    "M2": "MUSDB_LSB_2_Random_Fixed_DefaultSalt/CNN_D5_F64_LR0.0001_20260611-115403",
    "M3": "MUSDB_LSB_3_Random_Fixed_ContentSalt/CNN_D5_F64_LR0.0001_20260611-124709",
    "M4": "MUSDB_LSB_4_Random_Adaptive_DefaultSalt/CNN_D5_F64_LR0.0001_20260611-160127",
    "M5": "MUSDB_LSB5_Random_Adaptive_ContentSalt/CNN_D5_F64_LR0.0001_20260611-170236",
}


# experiments = [
#     { "label": "M1", "path": "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/Steganalysis/logs/MUSDB_LSB_1_NoRandom/CNN_D5_F64_LR0.0001_20260611-181303", "filter": "" },
#     { "label": "M2", "path": "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/Steganalysis/logs/MUSDB_LSB_2_Random_Fixed_DefaultSalt/CNN_D5_F64_LR0.0001_20260611-115403", "filter": "" },
#     { "label": "M3", "path": "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/Steganalysis/logs/MUSDB_LSB_3_Random_Fixed_ContentSalt/CNN_D5_F64_LR0.0001_20260611-124709", "filter": "" },
#     { "label": "M4", "path": "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/Steganalysis/logs/MUSDB_LSB_4_Random_Adaptive_DefaultSalt/CNN_D5_F64_LR0.0001_20260611-160127", "filter": "" },
#     { "label": "M5", "path": "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/Steganalysis/logs/MUSDB_LSB5_Random_Adaptive_ContentSalt/CNN_D5_F64_LR0.0001_20260611-170236", "filter": "" }
# ]


raw = {}
for method, folder in METHODS.items():
    aucs = []
    base_dir = os.path.join(BASE, folder)
    for sub in sorted(os.listdir(base_dir)):
        if not sub.startswith("run"):
            continue
        fpath = os.path.join(base_dir, sub, "experiment_results.csv")
        if os.path.isfile(fpath):
            df = pd.read_csv(fpath)
            if not df.empty:
                aucs.append(round(float(df.iloc[-1].get("Val_AUC", 0)), 4))
    raw[method] = aucs
    print(f"{method}: n={len(aucs)}, "
          f"mean={round(np.mean(aucs),4)}, "
          f"std={round(np.std(aucs,ddof=1),4)}")

M1: n=10, mean=0.8785, std=0.0183
M2: n=10, mean=0.7383, std=0.0101
M3: n=10, mean=0.7306, std=0.0134
M4: n=10, mean=0.7374, std=0.0148
M5: n=10, mean=0.7331, std=0.0099


In [ ]:
!pip install pingouin -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 3.2 MB/s eta 0:00:00


In [ ]:
import pingouin as pg
import numpy as np
import pandas as pd
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# ──  Mann-Whitney U + pingouin ─────────────────────────────
# Vallat, R. (2018). Pingouin: statistics in Python.
# Journal of Open Source Software, 3(31), 1026.
# https://doi.org/10.21105/joss.01026
#
# Mann-Whitney U: AUC bounded [0,1] → không Gaussian
# → nonparametric test bắt buộc (xem: Hanley & McNeil, 1982)
# Effect size: Rank-Biserial Correlation (Kerby, 2014)
# Multiple comparisons: Holm-Bonferroni (Holm, 1979)

def magnitude_r(r):
    r = abs(r)
    return "large" if r >= 0.5 else "medium" if r >= 0.3 else "small"

rows = []
for a, b in combinations(METHODS.keys(), 2):
    x = np.array(raw[a])
    y = np.array(raw[b])
    res = pg.mwu(x, y, alternative='two-sided')

    rows.append({
        "Pair":        f"{a} vs {b}",
        "Median_A":    round(np.median(x), 4),
        "Median_B":    round(np.median(y), 4),
        "U_val":       res['U_val'].values[0],
        "p_raw":       res['p_val'].values[0],
        "RBC":         round(res['RBC'].values[0], 4),
        "CLES":        round(res['CLES'].values[0], 4),
    })

df = pd.DataFrame(rows)
reject, p_adj, _, _ = multipletests(df['p_raw'], method='holm')
df['p_value']       = df['p_raw'].apply(lambda p: round(p, 4))
df['p_adj_holm']    = np.round(p_adj, 4)
df['Significant']   = reject
df['Effect_size']   = df['RBC'].apply(magnitude_r)

df = df.drop(columns=['p_raw'])
print(df.to_string(index=False))

Significance_dir = BASE + "/Significance-Tests"

df.to_csv(Significance_dir + "/mannwhitney_pingouin.csv", index=False)
print("\nSaved")

    Pair  Median_A  Median_B  U_val   RBC  CLES  p_value  p_adj_holm  Significant Effect_size
M1 vs M2    0.8726    0.7415  100.0  1.00 1.000   0.0002      0.0018         True       large
M1 vs M3    0.8726    0.7288  100.0  1.00 1.000   0.0002      0.0018         True       large
M1 vs M4    0.8726    0.7358  100.0  1.00 1.000   0.0002      0.0018         True       large
M1 vs M5    0.8726    0.7298  100.0  1.00 1.000   0.0002      0.0018         True       large
M2 vs M3    0.7415    0.7288   68.5  0.37 0.685   0.1735      1.0000        False      medium
M2 vs M4    0.7415    0.7358   55.0  0.10 0.550   0.7337      1.0000        False       small
M2 vs M5    0.7415    0.7298   66.0  0.32 0.660   0.2413      1.0000        False      medium
M3 vs M4    0.7288    0.7358   38.0 -0.24 0.380   0.3847      1.0000        False       small
M3 vs M5    0.7288    0.7298   43.0 -0.14 0.430   0.6232      1.0000        False       small
M4 vs M5    0.7358    0.7298   55.0  0.10 0.550   0.7337    

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations


BASE = "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/Steganalysis/logs"

METHODS = {
    "Alarood": "MUSDB_Alarood2022_CNN/CNN_D5_F64_LR0.0001_20260614-143241",
    "M5": "MUSDB_LSB5_Random_Adaptive_ContentSalt/CNN_D5_F64_LR0.0001_20260611-170236",
}



raw = {}
for method, folder in METHODS.items():
    aucs = []
    base_dir = os.path.join(BASE, folder)
    for sub in sorted(os.listdir(base_dir)):
        if not sub.startswith("run"):
            continue
        fpath = os.path.join(base_dir, sub, "experiment_results.csv")
        if os.path.isfile(fpath):
            df = pd.read_csv(fpath)
            if not df.empty:
                aucs.append(round(float(df.iloc[-1].get("Val_AUC", 0)), 4))
    raw[method] = aucs
    print(f"{method}: n={len(aucs)}, "
          f"mean={round(np.mean(aucs),4)}, "
          f"std={round(np.std(aucs,ddof=1),4)}")

Alarood: n=10, mean=0.7433, std=0.012
M5: n=10, mean=0.7331, std=0.0099


In [ ]:
import pingouin as pg
import numpy as np
import pandas as pd
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# ──  Mann-Whitney U + pingouin ─────────────────────────────
# Vallat, R. (2018). Pingouin: statistics in Python.
# Journal of Open Source Software, 3(31), 1026.
# https://doi.org/10.21105/joss.01026
#
# Mann-Whitney U: AUC bounded [0,1] → không Gaussian
# → nonparametric test bắt buộc (xem: Hanley & McNeil, 1982)
# Effect size: Rank-Biserial Correlation (Kerby, 2014)
# Multiple comparisons: Holm-Bonferroni (Holm, 1979)

def magnitude_r(r):
    r = abs(r)
    return "large" if r >= 0.5 else "medium" if r >= 0.3 else "small"

rows = []
for a, b in combinations(METHODS.keys(), 2):
    x = np.array(raw[a])
    y = np.array(raw[b])
    res = pg.mwu(x, y, alternative='two-sided')

    rows.append({
        "Pair":        f"{a} vs {b}",
        "Median_A":    round(np.median(x), 4),
        "Median_B":    round(np.median(y), 4),
        "U_val":       res['U_val'].values[0],
        "p_raw":       res['p_val'].values[0],
        "RBC":         round(res['RBC'].values[0], 4),
        "CLES":        round(res['CLES'].values[0], 4),
    })

df = pd.DataFrame(rows)
reject, p_adj, _, _ = multipletests(df['p_raw'], method='holm')
df['p_value']       = df['p_raw'].apply(lambda p: round(p, 4))
df['p_adj_holm']    = np.round(p_adj, 4)
df['Significant']   = reject
df['Effect_size']   = df['RBC'].apply(magnitude_r)

df = df.drop(columns=['p_raw'])
print(df.to_string(index=False))

Significance_dir = BASE + "/Significance-Tests"

df.to_csv(Significance_dir + "/mannwhitney_pingouin.csv", index=False)
print("\nSaved")

         Pair  Median_A  Median_B  U_val  RBC  CLES  p_value  p_adj_holm  Significant Effect_size
Alarood vs M5    0.7458    0.7298   74.0 0.48  0.74   0.0757      0.0757        False      medium

Saved
